In [ ]:
import pandas as pd
import geopandas as gpd
from os import path, environ, makedirs
from dotenv import load_dotenv
from unidecode import unidecode

from core.geo import areal_weighted_interpolation

In [ ]:
load_dotenv()

# Carregando os dados extraídos no notebook anterior

Neste notebook, vamos utilizar os dados extraídos e salvos pelo notebook `03 habitação - extração.ipynb`.

In [ ]:
cache_dir = path.join('data', 'cache', 'urbanismo')

In [ ]:
filename = path.join(cache_dir, 'orcamento_urbanismo_original.csv')
df_orcamento = pd.read_csv(filename,
            sep=';',
            decimal=',',
            encoding='utf8',
            dtype=str)
df_orcamento

In [ ]:
for col in [col for col in df_orcamento.columns if 'Vl' in col]:
    df_orcamento[col] = df_orcamento[col].astype(float)
df_orcamento['DataExtracao'] = pd.to_datetime(df_orcamento['DataExtracao'])
df_orcamento

In [ ]:
filename = path.join(cache_dir, 'orcamento_regionalizado_urbanismo_original.csv')
df_orcamento_r = pd.read_csv(filename,
            sep=';',
            decimal=',',
            encoding='utf8',
            dtype=str)
df_orcamento_r

In [ ]:
df_orcamento_r['VALOR_DETALHAMENTO_AÇÃO'] = df_orcamento_r['VALOR_DETALHAMENTO_AÇÃO'].astype(float)
df_orcamento_r

In [ ]:
df_orcamento_r.dtypes

## CSV de Subprefeituras do Qlik

In [ ]:
url_subs = environ.get('CSV_SUBPREFEITURAS_QLIK')
df_subs = pd.read_csv(url_subs)
df_subs

In [ ]:
df_subs = df_subs[['sub.CODIGO', 'sub.NOME']]
df_subs

In [ ]:
subs_qlik = df_subs['sub.NOME'].unique().tolist()
subs_qlik.sort()
subs_qlik

## Chave composta subprefeitura-ano

Como 3 tabelas possuem valores para mais de um ano, também vale a pena a criação de uma chave composta entre subprefeitura e ano. A tabela que possui mais períodos é a tabela da meta 12 do Programa de Metas, com os anos de 2021, 2022, 2023 e 2024. Vamos criar uma tabela com o produto cartesiano entre subprefeituras e anos.

In [ ]:
df_subs_ano = (
    df_subs[['sub.NOME']]
    .merge(pd.Series(data=[2024, 2025], name='ano'),
           how='cross')
)

df_subs_ano.loc[:, 'subprefeitura-ano'] = (
    df_subs_ano.loc[:, 'sub.NOME'] + ' | ' + df_subs_ano.loc[:, 'ano'].astype(str)
)

df_subs_ano

# Padronização de dados de Orçamento

Para o orçamento, além de padronizar os nomes de subprefeituras e tipos de dados das métricas, precisaremos também adaptar os dados para compatibilizar o orçamento regionalizado e não realizado. Para isso, vamos fazer o seguinte:

1. Classificar o orçamento detalhado por nível de regionalização nas seguintes categorias: subprefeitura, região e não regionalizável;
1. Agrupar o restante do orçamento não detalhado e manter apenas o orçamento inicial, atualizado e liquidado;
1. Subtrair o total do orçamento detalhado do orçamento não detalhado e classificar o nível de regionalização como não regionalizado;
1. Unir os dois dataframes de orçamento de acordo com as dimensões mantidas.

## Orçamento regionalizado

In [ ]:
df_orcamento_r.head(1)

In [ ]:
cols_orcamento_r = ['CÓDIGO_ÓRGÃO', 'SIGLA_ÓRGÃO', 'DESCRIÇÃO_ÓRGÃO',
                    'CÓDIGO_FUNÇÃO', 'DESCRIÇÃO_FUNÇÃO',
                    'CÓDIGO_PROGRAMA', 'DESCRIÇÃO_PROGRAMA',
                    'CÓDIGO_PROJ_ATIV', 'DESCRIÇÃO_PROJ_ATIV',
                    'CÓDIGO_VÍNCULO_PMSP', 'REGIÃO',
                    'SUBPREFEITURA', 'TIPO_REGIONALIZAÇÃO', 'ANO']

cols_orcamento_r_vl = ['VALOR_DETALHAMENTO_AÇÃO']

df_orcamento_r = df_orcamento_r[cols_orcamento_r + cols_orcamento_r_vl]
df_orcamento_r

In [ ]:
df_orcamento_r['TIPO_REGIONALIZAÇÃO'].value_counts()

In [ ]:
df_orcamento_r.loc[df_orcamento_r['TIPO_REGIONALIZAÇÃO'].isna(), 'TIPO_REGIONALIZAÇÃO'] = 'Despesa Não-Regionalizável'
df_orcamento_r

In [ ]:
df_orcamento_r['TIPO_REGIONALIZAÇÃO'].value_counts()

In [ ]:
df_orcamento_r = df_orcamento_r.groupby(cols_orcamento_r).sum().round(2).reset_index()

df_orcamento_r

In [ ]:
df_orcamento_r.loc[
    ~df_orcamento_r['REGIÃO'].str.contains('Supra', na=False),
    'NIVEL_REGIONALIZAÇÃO'] = 'Região'

df_orcamento_r

In [ ]:
df_orcamento_r.loc[
    ~df_orcamento_r['SUBPREFEITURA'].str.contains('Supra', na=False),
    'NIVEL_REGIONALIZAÇÃO'] = 'Subprefeitura'

df_orcamento_r

In [ ]:
df_orcamento_r.loc[df_orcamento_r['NIVEL_REGIONALIZAÇÃO'].isna(), 'NIVEL_REGIONALIZAÇÃO'] = 'Não regionalizável'
df_orcamento_r

## Orçamento não regionalizado

In [ ]:
df_orcamento.head(1)

In [ ]:
cols_orcamento = ['Cd_Orgao', 'Sigla_Orgao', 'Ds_Orgao', 'Cd_Programa',
                  'Cd_Funcao', 'Ds_Funcao',
                  'Ds_Programa', 'ProjetoAtividade', 'Ds_Projeto_Atividade',
                  'COD_VINC_REC_PMSP', 'ANO']

cols_orcamento_vl = ['Vl_Orcado_Ano', 'Vl_Orcado_Atualizado', 'Vl_Liquidado']

df_orcamento_original = df_orcamento.copy()
df_orcamento = df_orcamento[cols_orcamento + cols_orcamento_vl]
df_orcamento

In [ ]:
df_orcamento = df_orcamento.groupby(cols_orcamento).sum().reset_index()
df_orcamento

In [ ]:
r_agg_cols = ['CÓDIGO_ÓRGÃO', 'CÓDIGO_PROGRAMA', 'CÓDIGO_PROJ_ATIV',
              'CÓDIGO_FUNÇÃO', 'CÓDIGO_VÍNCULO_PMSP', 'ANO']

cols_to_drop = r_agg_cols.copy()
cols_to_drop.remove('ANO')

agg_cols = ['Cd_Orgao', 'Cd_Programa', 'ProjetoAtividade',
            'Cd_Funcao', 'COD_VINC_REC_PMSP', 'ANO']

df_orcamento_r_agg = (
    df_orcamento_r[r_agg_cols + ['VALOR_DETALHAMENTO_AÇÃO']]
    .groupby(r_agg_cols)
    .sum()
    .reset_index()
)

df_orcamento_r_agg.loc[:, 'VALOR_DETALHAMENTO_AÇÃO'] = (
    df_orcamento_r_agg
    .loc[:, 'VALOR_DETALHAMENTO_AÇÃO']
    .round(2)
)

df_orcamento_ajustado = df_orcamento.merge(
    df_orcamento_r_agg,
    left_on=agg_cols,
    right_on=r_agg_cols,
    how='left'
).drop(columns=cols_to_drop)

df_orcamento_ajustado

In [ ]:
df_orcamento_ajustado.loc[df_orcamento_ajustado['VALOR_DETALHAMENTO_AÇÃO'].isna(), 'VALOR_DETALHAMENTO_AÇÃO'] = 0

df_orcamento_ajustado.loc[:, 'Vl_Liquidado_N_Detalhado'] = (
    df_orcamento_ajustado.loc[:, 'Vl_Liquidado']
    - df_orcamento_ajustado.loc[:, 'VALOR_DETALHAMENTO_AÇÃO']).round(2)

df_orcamento_ajustado

In [ ]:
df_orcamento_ajustado[df_orcamento_ajustado['Vl_Liquidado_N_Detalhado']<0]

In [ ]:
df_orcamento_ajustado[['Vl_Liquidado', 'VALOR_DETALHAMENTO_AÇÃO']].sum()

## Unindo os dados de orçamento

Agora, vamos adicionar os dados não detalhados ao dataframe que contém o orçamento detalhado.

In [ ]:
orcamento_cols_map = {'Cd_Orgao': 'CÓDIGO_ÓRGÃO',
                      'Sigla_Orgao': 'SIGLA_ÓRGÃO',
                      'Ds_Orgao': 'DESCRIÇÃO_ÓRGÃO',
                      'Cd_Funcao': 'CÓDIGO_FUNÇÃO',
                      'Ds_Funcao': 'DESCRIÇÃO_FUNÇÃO',
                      'Cd_Programa': 'CÓDIGO_PROGRAMA',
                      'Ds_Programa': 'DESCRIÇÃO_PROGRAMA',
                      'ProjetoAtividade': 'CÓDIGO_PROJ_ATIV',
                      'Ds_Projeto_Atividade': 'DESCRIÇÃO_PROJ_ATIV',
                      'COD_VINC_REC_PMSP': 'CÓDIGO_VÍNCULO_PMSP',
                      'Vl_Liquidado_N_Detalhado': 'Vl_Liquidado'}

df_orcamento_ajustado = (
    df_orcamento_ajustado
    .drop(columns=['VALOR_DETALHAMENTO_AÇÃO', 'Vl_Liquidado'])
    .rename(columns=orcamento_cols_map)
    )

df_orcamento_ajustado

In [ ]:
df_orcamento_r = (df_orcamento_r
                  .rename(columns={'VALOR_DETALHAMENTO_AÇÃO': 'Vl_Liquidado'}))

df_orcamento_r

In [ ]:
df_orcamento_final = pd.concat([df_orcamento_r, df_orcamento_ajustado])

df_orcamento_final

In [ ]:
df_orcamento_final.loc[df_orcamento_final['Vl_Orcado_Ano'].isna(),
                       'Vl_Orcado_Ano'] = 0
df_orcamento_final.loc[df_orcamento_final['Vl_Orcado_Atualizado'].isna(),
                       'Vl_Orcado_Atualizado'] = 0
df_orcamento_final.loc[df_orcamento_final['NIVEL_REGIONALIZAÇÃO'].isna(),
                       'NIVEL_REGIONALIZAÇÃO'] = 'Não detalhado'

df_orcamento_final

## Padronizando os nomes de subprefeituras

In [ ]:
subs_orcamento = (
    df_orcamento_final.loc[~df_orcamento_final['SUBPREFEITURA'].isna(), 'SUBPREFEITURA']
    .apply(unidecode)
    .unique()
    .tolist()
)

subs_orcamento.sort()

subs_orcamento

In [ ]:
subs_orcamento[:-6]

In [ ]:
len(subs_orcamento[:-6])

In [ ]:
subs_qlik_orcamento = subs_qlik.copy()
# Guainases e Vila Prudente aparecem em uma ordenação diferente, por isso serão
# removidas e adicionadas novamente ao final da lista
subs_qlik_orcamento.remove('GUAIANASES')
subs_qlik_orcamento.remove('VILA PRUDENTE')
subs_qlik_orcamento.append('GUAIANASES')
subs_qlik_orcamento.append('VILA PRUDENTE')

In [ ]:
mapper_orcamento = {
    so: sq
    for so, sq in zip(subs_orcamento, subs_qlik_orcamento)
}

mapper_orcamento

In [ ]:
df_orcamento_final.insert(
    7,
    'sub.NOME',
    df_orcamento_final.loc[:,'SUBPREFEITURA'].apply(lambda s: unidecode(s) if isinstance(s, str) else None).map(mapper_orcamento)
)

df_orcamento_final

## Adicionando as subprefeituras faltantes

In [ ]:
orcamento_subs_cols = ['CÓDIGO_ÓRGÃO', 'SIGLA_ÓRGÃO', 'DESCRIÇÃO_ÓRGÃO',
                       'CÓDIGO_FUNÇÃO', 'DESCRIÇÃO_FUNÇÃO',
                       'CÓDIGO_PROGRAMA', 'DESCRIÇÃO_PROGRAMA',
                       'CÓDIGO_PROJ_ATIV', 'DESCRIÇÃO_PROJ_ATIV',
                       'CÓDIGO_VÍNCULO_PMSP', 'ANO']

df_orcamento_subs = df_orcamento_final[orcamento_subs_cols].copy()
df_orcamento_subs = df_orcamento_subs.drop_duplicates().reset_index(drop=True)
df_orcamento_subs

In [ ]:
df_orcamento_subs['TIPO_REGIONALIZAÇÃO'] = 'Despesa Regionalizável'
df_orcamento_subs['NIVEL_REGIONALIZAÇÃO'] = 'Subprefeitura'

df_orcamento_subs

In [ ]:
df_orcamento_subs = (
    df_orcamento_subs
    .merge(pd.DataFrame(columns=['sub.NOME'], data=subs_qlik),
           how='cross')
)

df_orcamento_subs

In [ ]:
df_orcamento_completo = (
    df_orcamento_final
    .merge(df_orcamento_subs,
           how='outer',
           on=df_orcamento_subs.columns.tolist())
)

df_orcamento_completo

In [ ]:
df_orcamento_completo.loc[df_orcamento_completo['Vl_Liquidado'].isna(), 'Vl_Liquidado'] = 0
df_orcamento_completo.loc[df_orcamento_completo['Vl_Orcado_Ano'].isna(), 'Vl_Orcado_Ano'] = 0
df_orcamento_completo.loc[df_orcamento_completo['Vl_Orcado_Atualizado'].isna(), 'Vl_Orcado_Atualizado'] = 0

df_orcamento_completo

## Adicionando descrição das vinculações

In [ ]:
df_orcamento_completo = (
    df_orcamento_completo
    .merge(df_orcamento_original[['COD_VINC_REC_PMSP', 'TXT_VINC_PMSP']].drop_duplicates(),
            how='left',
            left_on='CÓDIGO_VÍNCULO_PMSP',
            right_on='COD_VINC_REC_PMSP')
    .drop(columns='COD_VINC_REC_PMSP')
)

df_orcamento_completo

## Adicionando a chave composta subprefeitura-ano

Antes de unir as tabelas, precisamos converter o ano da tabela de orçamento em número inteiro.

In [ ]:
df_orcamento_completo['ANO'] = df_orcamento_completo['ANO'].astype(int)
df_orcamento_completo

In [ ]:
df_orcamento_completo = df_orcamento_completo.merge(df_subs_ano,
                                                    how='left',
                                                    left_on=['sub.NOME', 'ANO'],
                                                    right_on=['sub.NOME', 'ano'])

df_orcamento_completo

# Separar tabelas de orçamento por tema

Os dados de orçamento são apresentados em 3 dashboards temáticos diferentes: um sobre gestão dos riscos ambientais; outro sobre implantação e manutenção de parques; e, função habitação em geral. Existe um quarto dashboard relacionado a orçamento que trata de todo o PPA, mas seus dados não vem exatamente desses mesmos arquivos.

Portanto, para os 3 dashboards que utilizam os dados de orçamento tratados aqui, precisamos segmentar as tabelas com o conteúdo específico de cada dashboard.

In [ ]:
df_orcamento_riscos = df_orcamento_completo.loc[
    df_orcamento_completo['CÓDIGO_PROGRAMA']=='3008'
    ]

df_orcamento_riscos

In [ ]:
df_orcamento_parques = df_orcamento_completo.loc[
    df_orcamento_completo['CÓDIGO_PROJ_ATIV'].isin(['1702', '1703'])
]
df_orcamento_parques

In [ ]:
df_orcamento_habitacao = df_orcamento_completo.loc[
    df_orcamento_completo['CÓDIGO_FUNÇÃO']=='16'
]

df_orcamento_habitacao

# Armazenamento

Finalmente, vamos exportar os dados em formato csv compatível com o Qlik e no padrão do excel para português do Brasil.

In [ ]:
base_path = path.join('data_output', 'urbanismo')

if not path.exists(base_path):
    makedirs(base_path)

for name, df in [('orcamento-riscos', df_orcamento_riscos),
                 ('orcamento-parques', df_orcamento_parques),
                 ('orcamento-habitacao', df_orcamento_habitacao),]:

    filepath = path.join(base_path, f'{name}.csv')

    df.to_csv(filepath,
              index=False,
              sep=';',
              decimal=',',
              encoding='latin1')